# Create & Plot Embeddings
Loads raw strain h5 → preprocess (bandpass + whiten) → embed model → side-by-side plot of raw / whitened strain and corresponding embeddings.

Follows the same pipeline as `scripts/compute_embeddings.py`.

In [1]:
# ── USER CONFIG ──────────────────────────────────────────────────────────────
import sys, os
# Set repo root so gwak/ml4gw are importable
REPO_ROOT = os.path.abspath('../../..')   # gwak/output/embeddings -> repo root
sys.path.insert(0, REPO_ROOT)

MODEL_PATH  = f'{REPO_ROOT}/gwak/output/ResNet_HL/model_JIT.pt'
STRAIN_FILE = f'{REPO_ROOT}/scripts/burst_benchmark_short-0.h5'  # raw strain h5
IFOS        = ['H1', 'L1']

# Preprocessing params (must match training)
SAMPLE_RATE    = 4096
KERNEL_LENGTH  = 1.0    # seconds per window
PSD_LENGTH     = 64.0   # seconds used for PSD estimation
FDURATION      = 1.0    # whitening duration
FFTLENGTH      = 2.0    # FFT length for spectral density
HIGHPASS       = 30.0   # Hz

# How many consecutive 1-second windows to visualise
N_WINDOWS_PLOT = 10

print('Config OK')

Config OK


## 1. Imports & model

In [2]:
import h5py
import numpy as np
import torch
import matplotlib.pyplot as plt
from pathlib import Path

from ml4gw.transforms import SpectralDensity, Whiten
from gwak.train.dataloader import TorchBandpassFIR

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

# Load JIT embedding model
model = torch.jit.load(MODEL_PATH, map_location=device)
model.eval()
print('Model loaded:', MODEL_PATH)

ModuleNotFoundError: No module named 'ml4gw.transforms'

## 2. Preprocessing transforms

In [ ]:
bandpass = TorchBandpassFIR(
    lowcut=HIGHPASS,
    highcut=2047,
    sample_rate=SAMPLE_RATE
).to(device)

whitener = Whiten(
    FDURATION,
    SAMPLE_RATE,
    highpass=HIGHPASS,
).to(device)

spectral_density = SpectralDensity(
    SAMPLE_RATE,
    FFTLENGTH,
    average='median'
).to(device)

print('Transforms ready')

## 3. Load raw strain data

In [ ]:
def read_strain(h5_path, ifos):
    with h5py.File(h5_path, 'r') as f:
        print('Keys in file:', list(f.keys()))
        strain = {}
        for ifo in ifos:
            if ifo in f:
                strain[ifo] = f[ifo][:]
                print(f'  {ifo}: {strain[ifo].shape}, dtype={strain[ifo].dtype}')
            else:
                print(f'  WARNING: {ifo} not found')
    return strain

strain_data = read_strain(STRAIN_FILE, IFOS)

# Stack to [num_ifos, total_samples]
continuous = np.stack([strain_data[ifo] for ifo in IFOS], axis=0)
print('continuous shape:', continuous.shape)
total_samples = continuous.shape[1]
print(f'Total duration: {total_samples / SAMPLE_RATE:.1f} s')

## 4. Extract windows, preprocess, embed

In [ ]:
kernel_samples   = int(KERNEL_LENGTH * SAMPLE_RATE)
psd_samples      = int(PSD_LENGTH    * SAMPLE_RATE)
fduration_samples = int(FDURATION    * SAMPLE_RATE)

# Find first valid start: need PSD_LENGTH of data before the window
first_start = psd_samples   # window starts here, PSD covers [0, psd_samples)

raw_windows      = []  # [N, num_ifos, kernel_samples]  — raw (unwhitened) strain
whitened_windows = []  # [N, num_ifos, kernel_samples]  — after preprocessing
window_times_s   = []  # start time in seconds for each window

with torch.no_grad():
    for i in range(N_WINDOWS_PLOT):
        start = first_start + i * kernel_samples
        end   = start + kernel_samples + fduration_samples

        if end > total_samples:
            print(f'Not enough data for window {i}, stopping at {i}')
            break

        # ---- raw (just for plotting) ----
        raw_seg = continuous[:, start:start + kernel_samples]   # [C, T]
        raw_windows.append(raw_seg)
        window_times_s.append(start / SAMPLE_RATE)

        # ---- PSD region ----
        psd_seg  = torch.tensor(continuous[:, start - psd_samples:start],
                                dtype=torch.float32, device=device).unsqueeze(0)

        # ---- data to whiten: kernel + fduration ----
        data_seg = torch.tensor(continuous[:, start:end],
                                dtype=torch.float32, device=device).unsqueeze(0)

        # bandpass
        data_bp = bandpass(data_seg)

        # PSD
        psds = spectral_density(psd_seg.double())

        # whiten
        whitened = whitener(data_bp.double(), psds.double()).float()

        # keep only kernel portion
        whitened = whitened[:, :, :kernel_samples]  # [1, C, T]

        whitened_windows.append(whitened[0].cpu().numpy())  # [C, T]

raw_windows      = np.array(raw_windows)       # [N, C, T]
whitened_windows = np.array(whitened_windows)  # [N, C, T]
window_times_s   = np.array(window_times_s)

print(f'Extracted {len(raw_windows)} windows')
print('raw shape:', raw_windows.shape)
print('whitened shape:', whitened_windows.shape)

In [ ]:
# Compute embeddings for all whitened windows
with torch.no_grad():
    batch = torch.tensor(whitened_windows, dtype=torch.float32, device=device)
    embeddings = model(batch).cpu().numpy()  # [N, embed_dim]

print('embeddings shape:', embeddings.shape)

## 5. Plot: raw strain → whitened strain → embedding

In [ ]:
t_kernel = np.linspace(0, KERNEL_LENGTH, kernel_samples)
n_ifo    = len(IFOS)
n_rows   = n_ifo * 2 + 1   # raw H1, raw L1, whitened H1, whitened L1, embeddings

for win_idx in range(len(raw_windows)):
    t_offset = window_times_s[win_idx]
    fig, axes = plt.subplots(n_rows, 1, figsize=(14, 2.5 * n_rows), sharex=False)
    fig.suptitle(f'Window {win_idx}  (t₀ = {t_offset:.1f} s)', fontsize=13)

    row = 0
    # Raw strain
    for ch, ifo in enumerate(IFOS):
        axes[row].plot(t_kernel, raw_windows[win_idx, ch], linewidth=0.6, color='steelblue')
        axes[row].set_ylabel(f'{ifo} raw')
        axes[row].set_xlabel('time (s)')
        row += 1

    # Whitened strain
    for ch, ifo in enumerate(IFOS):
        axes[row].plot(t_kernel, whitened_windows[win_idx, ch], linewidth=0.6, color='darkorange')
        axes[row].set_ylabel(f'{ifo} whitened')
        axes[row].set_xlabel('time (s)')
        row += 1

    # Embedding (one bar per dim)
    emb = embeddings[win_idx]
    axes[row].bar(range(len(emb)), emb, color='mediumseagreen')
    axes[row].set_xlabel('embedding dimension')
    axes[row].set_ylabel('value')
    axes[row].set_title('embedding')

    plt.tight_layout()
    plt.show()

## 6. All windows: embedding dims over time

In [ ]:
# Concatenate whitened strain into one long timeseries for plotting
long_whitened = whitened_windows.reshape(len(whitened_windows) * kernel_samples, len(IFOS), order='C')
# Actually reshape properly: [N, C, T] -> [C, N*T]
long_whitened = whitened_windows.transpose(1, 0, 2).reshape(len(IFOS), -1)  # [C, N*T]
t_long = np.arange(long_whitened.shape[1]) / SAMPLE_RATE

embed_dim = embeddings.shape[1]
n_dims_show = min(6, embed_dim)

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=False)

# Panel 1: H1 whitened
axes[0].plot(t_long, long_whitened[0], linewidth=0.5, color='steelblue')
axes[0].set_ylabel('H1 whitened strain')
axes[0].set_xlabel('time within segment (s)')

# Panel 2: L1 whitened
axes[1].plot(t_long, long_whitened[1], linewidth=0.5, color='darkorange')
axes[1].set_ylabel('L1 whitened strain')
axes[1].set_xlabel('time within segment (s)')

# Panel 3: embedding dims over window index
for d in range(n_dims_show):
    axes[2].plot(window_times_s, embeddings[:, d], marker='o', markersize=4, label=f'dim {d}')
axes[2].set_xlabel('time (s)')
axes[2].set_ylabel('embedding dims readout')
axes[2].set_title('Embedding dimensions over time (1 point per 1-second window)')
axes[2].legend()

plt.tight_layout()
plt.show()